In [ ]:
# >>> repo path setup (auto-added during reorg; keeps this notebook runnable from notebooks/) <<<
import os, sys
_p = os.getcwd()
while not os.path.isdir(os.path.join(_p, 'src')) and os.path.dirname(_p) != _p:
    _p = os.path.dirname(_p)
os.chdir(_p)
sys.path.insert(0, os.path.join(_p, 'src'))
# <<< end repo path setup >>>

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from tqdm import tqdm
from pathlib import Path

from seisfwi.utils import interp_data2d
from seisfwi.model import RockPhysicsGassmann

from utils import project_plots
from utils import RockPhysicsModel

In [ ]:
project_plots.configure_plot_settings()

### Extract 2D Slices from 3D Saturation Models

In [ ]:
path_3D = Path('/net/vision/scr2/haipeng/FWI-HMC/model3D')
path_2D = Path('/net/vision/scr2/haipeng/FWI-HMC/model2D')
file_list = os.listdir(path_3D)
file_list = [file for file in file_list if file.startswith('flow_1year') and not file.startswith('flow_1year_128_128_35_1well_10012_to_10012')]
# file_list

In [ ]:
file = file_list[2]
with h5py.File(path_3D/file, 'r') as f:
    pressure = f['pressure'][:]

In [ ]:
pressure.shape

In [ ]:
plt.imshow(pressure[0, 10, :, 64])
plt.colorbar()

In [ ]:
# time_indices = [10, 20, 25, 30] # [10]
# saturation_all_x = {t : [] for t in time_indices}
# saturation_all_y = {t : [] for t in time_indices}

# for file in file_list:
#     print(f'Processing {file}')
    
#     with h5py.File(path_3D/file, 'r') as f:
#         saturation = f['saturation'][:]
        

In [ ]:
# time_indices = [10, 20, 25, 30] # [10]
# saturation_all_x = {t : [] for t in time_indices}
# saturation_all_y = {t : [] for t in time_indices}

# for file in file_list:
#     print(f'Processing {file}')
    
#     with h5py.File(path_3D/file, 'r') as f:
#         saturation = f['saturation'][:]
        
#         for t in time_indices:
#             saturation_all_x[t].append(saturation[:, t, :, 64, :])
#             saturation_all_y[t].append(saturation[:, t, :, :, 64])

# for t in time_indices:
#     saturation_all_x[t] = np.concatenate(saturation_all_x[t], axis=0)
#     saturation_all_y[t] = np.concatenate(saturation_all_y[t], axis=0)
    
#     # Save original prior samples
#     saturation_all = np.concatenate((saturation_all_y[t], saturation_all_x[t]), axis=0)
#     np.save(path_2D / f"sa_prior_time{t}_slice64_8002_samples_orig.npy", saturation_all)
    
#     # Interpolate all prior samples
#     saturation_all = np.array([interp_data2d(sample, 2, 7, 5, method='linear') for sample in tqdm(saturation_all, desc='Interpolating priors')])
#     np.save(path_2D / f"sa_prior_time{t}_slice64_8002_samples_5m.npy", saturation_all)


# poro   = np.load(path_3D / 'poro0.npy')
# poro_x = poro[:, :, 64, :]
# poro_y = poro[:, :, :, 64]

# # sort files to ensure correct order
# # porosity
# file_indices = [tuple(map(int, re.search(r'_(\d+)_to_(\d+)_', f).groups())) for f in file_list]
# poro_x = np.concatenate([poro_x[start-1:end] for start, end in file_indices])
# poro_y = np.concatenate([poro_y[start-1:end] for start, end in file_indices])
# poro   = np.concatenate((poro_y, poro_x), axis=0)
# poro   = np.array([interp_data2d(sample, 2, 7, 5, method='linear') for sample in tqdm(poro, desc='Interpolating priors')])
# np.save(path_2D / f"po_prior_slice64_8002_samples_5m.npy", poro)

### Saturation and Porosity

In [ ]:
id = 800
po_co2 = np.load(path_2D / f'po_prior_slice64_8002_samples_5m.npy')[id]
sa_co2 = np.load(path_2D / f'sa_prior_time25_slice64_8002_samples_5m.npy')[id]

nz0 = 260
nx0 = 110
nz_res = po_co2.shape[0]
nx_res = po_co2.shape[1]
print(nx_res, nz_res)

### Baseline Model

In [ ]:
# load background velocity model
vp_bl = np.load("model/vp_nx501_ny401_nz481_dx20m_dy20m_dz5m.npy")[:, 200, :]
vp_bl = interp_data2d(vp_bl, 10, 5, 7, method='linear').T
vp_bl = vp_bl[5:351, 100:501] * 0.8
vp_bl[vp_bl<2000] = 2000.0

# scale the velocity
scale = np.exp(-0.8 * (po_co2 - po_co2.mean()))
vp_bl[nz0:nz0+nz_res, nx0:nx0+nx_res] *= scale

# reservoir model
vp_res = vp_bl[nz0:nz0+nz_res, nx0:nx0+nx_res].copy()

# axis
nz = vp_bl.shape[0]
nx = vp_bl.shape[1]

x = np.arange(nx) * 5.0
z = np.arange(nz) * 5.0
x_res = (np.arange(nx_res) + nx0) * 5.0
z_res = (np.arange(nz_res) + nz0) * 5.0 

In [ ]:
z_res

In [ ]:
project_plots.plot_2d(x, z, vp_bl, vmin=2000, vmax=5000, cmap='jet', figsize = (5, 6))

In [ ]:
project_plots.plot_2d(x_res, z_res, vp_res, vmin=2000, vmax=5000, cmap='jet', figsize = (10, 4))

### Rock Physics Models

In [ ]:
# Parameters for rock physics model
rock_physics_params = RockPhysicsModel(vp_res)

# Gassmann fluid substitution
rockphy = RockPhysicsGassmann(**rock_physics_params)


vs_res = rock_physics_params["vs_res"]
rho_res = rock_physics_params["rho_res"]

In [ ]:
vp_co2, vs_co2, rho_co2 = rockphy(sa_co2)
vp_co2 = vp_co2.cpu().numpy()
vs_co2 = vs_co2.cpu().numpy()
rho_co2 = rho_co2.cpu().numpy()

In [ ]:
models = [
    (vp_res,   "Baseline Vp"),
    (vp_co2,   "Vp Change"),
    (vs_res,   "Baseline Vs"),
    (vs_co2,   "Vs Change"),
    (rho_res,  "Baseline Density"),
    (rho_co2,  "Density Change")
]

fig, axes = plt.subplots(3, 2, figsize=(15, 8), dpi=120)

for ax, (data, title) in zip(axes.ravel(), models):
    im = ax.imshow(data, aspect='auto',
                   extent=[x_res[0], x_res[-1], z_res[-1], z_res[0]],
                   cmap='jet')
    fig.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Distance (m)")
    ax.set_ylabel("Depth (m)")
    ax.grid(linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### Save Target Model

In [ ]:
vp_ml = vp_bl.copy()
vp_ml[nz0:nz0+nz_res, nx0:nx0+nx_res] += vp_co2

In [ ]:
np.save(f'model/vp_bl_nz{nz}_nx{nx}_5m.npy', vp_bl)
np.save(f'model/vp_ml_nz{nz}_nx{nx}_5m.npy', vp_ml)
np.save(f'model/po_res_nz{nz_res}_nx{nx_res}_5m.npy', po_co2)
np.save(f'model/sa_res_nz{nz_res}_nx{nx_res}_5m.npy', sa_co2)
np.save(f'model/vp_res_nz{nz_res}_nx{nx_res}_5m.npy', vp_res)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2), dpi=150)
for ax, data, title in zip(axes, [po_co2, sa_co2, vp_co2], ['Porosity', 'SCO2', 'Vp (m/s)']):
    ax.imshow(data, cmap='jet', aspect='auto', extent=(x_res[0], x_res[-1], z_res[-1], z_res[0]))
    ax.set_title(title)
    ax.set_ylabel('Depth (m)')
    ax.set_xlabel('Distance (m)')
    ax.grid(True, linestyle="--", alpha=0.5)
    cbar = plt.colorbar(ax.images[0], ax=ax, orientation='vertical', pad=0.02)
    cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
project_plots.plot_2d(x, z, vp_ml - vp_bl, vmin=-200, vmax=200, cmap='jet', xlim=(500, 1500), ylim=(1400, 1200))

In [ ]:
fontsize = 12

plt.rcParams.update({
    "font.size": fontsize,
    "axes.labelsize": fontsize,
    "axes.titlesize": fontsize,
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})


extent     = [x[0]/1000, x[-1]/1000, z[-1]/1000, z[0]/1000]
extent_res = [x_res[0]/1000, x_res[-1]/1000, (z_res[-1] - z_res[0]), 0]

In [ ]:
# --- Figure layout ---
fig = plt.figure(figsize=(10, 5), dpi=150)
gs = gridspec.GridSpec(3, 2, wspace=0.2, hspace=0.45)

# --- 1. Baseline Vp model ---
ax_bl = fig.add_subplot(gs[:, 0])
im_bl = ax_bl.imshow(vp_bl, cmap='jet', extent=extent, aspect='auto', vmin=1800, vmax=5800)

# Source and receiver geometry
src_x = np.array([400 * isrc + 200 for isrc in range(5)]) / 1000
src_z = np.full_like(src_x, 30) / 1000
rec_z = np.array([5.0 * (irec + 1) for irec in range(nz - 35)]) / 1000
rec_x = np.full_like(rec_z, 700) / 1000

ax_bl.scatter(src_x, src_z, marker='*', s=180, facecolor='red', edgecolor='k', linewidths=0.8, zorder=10, label="Seismic Source")
ax_bl.plot(rec_x, rec_z, linestyle='-', color='k', linewidth=1.5, zorder=5, label="Geophone well")
ax_bl.plot([1, 1], [0, 1.35], linestyle='--', color='#CC00FF', linewidth=1.2, zorder=5, label="Injection well")

# Reservoir box
rect = plt.Rectangle((x_res[0]/1000, z_res[0]/1000),
                     (x_res[-1] - x_res[0])/1000,
                     (z_res[-1] - z_res[0])/1000,
                     linewidth=0.8, edgecolor='black', facecolor='none', linestyle='--')
ax_bl.add_patch(rect)

# Axis styling
ax_bl.set_xlabel('Distance (km)')
ax_bl.set_ylabel('Depth (km)')
ax_bl.set_xticks([0, 0.5, 1.0, 1.5, 2.0])
ax_bl.set_yticks([0, 0.5, 1.0, 1.5])
ax_bl.minorticks_on()
ax_bl.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax_bl.grid(True, which='major', linestyle='--', linewidth=0.8, alpha=0.4, color='gray')
ax_bl.legend(loc='upper right', bbox_to_anchor=(1.0, 0.9), fontsize=10, framealpha=0.5)
ax_bl.text(0.05, 1.02, "a)", transform=ax_bl.transAxes, fontsize=19, fontweight="bold", va="bottom", ha="right")

# Small inset colorbar (inside subplot)
cb_ax = inset_axes(ax_bl, width="1.8%", height="28%", loc='lower right', bbox_to_anchor=(-0.16, 0.03, 1, 1), bbox_transform=ax_bl.transAxes, borderpad=0)
cb_bl = fig.colorbar(im_bl, cax=cb_ax, orientation='vertical')
cb_bl.set_ticks([2000, 3000, 4000, 5000])
cb_bl.set_label('Vp (m/s)', fontsize=10)
cb_bl.ax.tick_params(labelsize=10)

# --- 2. Porosity model ---
ax_po = fig.add_subplot(gs[0, 1])
im_po = ax_po.imshow(po_co2, aspect='auto', cmap='turbo', vmin=0, vmax=0.4, extent=extent_res)
ax_po.set_yticks([0, 25, 50])
ax_po.minorticks_on()
ax_po.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax_po.grid(True, which='major', linestyle='--', linewidth=0.8, alpha=0.4, color='gray')
ax_po.set_ylabel('Depth (m)')
cbar_po = fig.colorbar(im_po, ax=ax_po, orientation='vertical', fraction=0.045, pad=0.02)
cbar_po.set_label('Porosity', fontsize=10)
cbar_po.ax.tick_params(labelsize=10)
ax_po.text(-0.07, 1.04, "b)", transform=ax_po.transAxes, fontsize=19, fontweight="bold", va="bottom", ha="right")

# --- 3. Saturation model ---
ax_sa = fig.add_subplot(gs[1, 1])
im_sa = ax_sa.imshow(sa_co2, aspect='auto', cmap='inferno', vmin=0, vmax=0.5, extent=extent_res)
ax_sa.set_yticks([0, 25, 50])
ax_sa.minorticks_on()
ax_sa.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax_sa.grid(True, which='major', linestyle='--', linewidth=0.8, alpha=0.4, color='gray')
ax_sa.set_ylabel('Depth (m)')
cbar_sa = fig.colorbar(im_sa, ax=ax_sa, orientation='vertical', fraction=0.045, pad=0.02)
cbar_sa.set_label('Saturation', fontsize=10)
cbar_sa.ax.tick_params(labelsize=10)
cbar_sa.set_ticks([0, 0.25, 0.5])
ax_sa.text(-0.07, 1.04, "c)", transform=ax_sa.transAxes, fontsize=19, fontweight="bold", va="bottom", ha="right")


# --- 4. Velocity change model ---
ax_vp = fig.add_subplot(gs[2, 1])
im_vp = ax_vp.imshow(vp_co2, aspect='auto', cmap='jet', vmin=-240, vmax=240, extent=extent_res)
ax_vp.set_yticks([0, 25, 50])
ax_vp.minorticks_on()
ax_vp.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax_vp.grid(True, which='major', linestyle='--', linewidth=0.8, alpha=0.4, color='gray')
ax_vp.set_ylabel('Depth (m)')
ax_vp.set_xlabel('Distance (km)')
cbar_vp = fig.colorbar(im_vp, ax=ax_vp, orientation='vertical', fraction=0.045, pad=0.02)
cbar_vp.set_label('Vp (m/s)', fontsize=10)
cbar_vp.ax.tick_params(labelsize=10)
ax_vp.text(-0.07, 1.04, "d)", transform=ax_vp.transAxes, fontsize=19, fontweight="bold", va="bottom", ha="right")

# --- Adjust overall spacing ---
plt.subplots_adjust(left=0.08, right=0.96, top=0.95, bottom=0.08, wspace=0.3, hspace=0.45)
plt.savefig("./figures/model.png", dpi=300, bbox_inches='tight')
plt.show()

### Models for Error Analysis

In [ ]:
brie_component_new = 3
rock_physics_params.update({'brie_component': brie_component_new})
rockphy_new = RockPhysicsGassmann(**rock_physics_params)

In [ ]:
sa_co2_small = np.load(path_2D / f'sa_prior_time10_slice64_8002_samples_5m.npy')[id]
sa_co2_large = np.load(path_2D / f'sa_prior_time30_slice64_8002_samples_5m.npy')[id]

In [ ]:
# size
vp_co2_small, _, _ = rockphy(sa_co2_small)
vp_co2_large, _, _ = rockphy(sa_co2_large)

# different model
vp_co2_new, _, _ = rockphy_new(sa_co2)

vp_ml_small = vp_bl.copy()
vp_ml_large = vp_bl.copy()
vp_ml_new   = vp_bl.copy()

vp_ml_small[nz0:nz0+nz_res, nx0:nx0+nx_res] += vp_co2_small.cpu().numpy()
vp_ml_large[nz0:nz0+nz_res, nx0:nx0+nx_res] += vp_co2_large.cpu().numpy()
vp_ml_new[nz0:nz0+nz_res, nx0:nx0+nx_res]   += vp_co2_new.cpu().numpy()

In [ ]:
project_plots.plot_2d(x, z, vp_ml_small - vp_bl, vmin=-200, vmax=200, cmap='jet', xlim=(500, 1500), ylim=(1400, 1200))

In [ ]:
project_plots.plot_2d(x, z, vp_ml_large - vp_bl, vmin=-200, vmax=200, cmap='jet', xlim=(500, 1500), ylim=(1400, 1200))

In [ ]:
project_plots.plot_2d(x, z, vp_ml_new - vp_bl, vmin=-200, vmax=200, cmap='jet', xlim=(500, 1500), ylim=(1400, 1200))

In [ ]:
project_plots.plot_2d(x, z, vp_ml - vp_bl, vmin=-200, vmax=200, cmap='jet', xlim=(500, 1500), ylim=(1400, 1200))

In [ ]:
np.save(f'model/vp_ml_nz{nz}_nx{nx}_5m_new.npy',   vp_ml_new)
np.save(f'model/vp_ml_nz{nz}_nx{nx}_5m_small.npy', vp_ml_small)
np.save(f'model/vp_ml_nz{nz}_nx{nx}_5m_large.npy', vp_ml_large)

np.save(f'model/sa_res_nz{nz_res}_nx{nx_res}_5m_new.npy',   sa_co2)
np.save(f'model/sa_res_nz{nz_res}_nx{nx_res}_5m_small.npy', sa_co2_small)
np.save(f'model/sa_res_nz{nz_res}_nx{nx_res}_5m_large.npy', sa_co2_large)

In [ ]:
xind = 200

plt.figure(figsize=(10, 8), dpi=150)
plt.plot(vp_ml[:, xind]    - vp_bl[:,xind], z/1000, label='Used', linewidth=1.2)
plt.plot(vp_ml_new[:, xind]- vp_bl[:,xind], z/1000, label='New', linewidth=1.2)
plt.ylim(z_res[-1]/1000, z_res[0]/1000)
plt.xlabel('Vp (m/s)')
plt.ylabel('Depth (km)')
plt.legend()
plt.show()